# 06 · Streaming Anomaly Detection

## What this notebook covers
Batch anomaly detection assumes a fixed dataset. Production systems generate data continuously. This notebook implements online anomaly detection with concept drift handling — the key challenge being that "normal" can change over time.

## The concept drift problem
In a stream, the data distribution P(X) may shift:
- **Sudden drift**: a process change causes an abrupt distribution shift
- **Gradual drift**: slow evolution of normal behaviour
- **Recurring drift**: seasonal patterns return

A model trained once on historical data will degrade as drift accumulates. Online detectors must adapt: either by continuous partial refit or by explicit drift detection followed by model reset.

## Methods implemented
- **Online Isolation Forest** with sliding window retraining
- **ADWIN** (Adaptive Windowing): a rigorous online drift detector that maintains a variable-length window and triggers when the mean of two sub-windows diverges beyond a statistical threshold. It is the reference algorithm for streaming drift detection.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from collections import deque
import matplotlib.pyplot as plt
np.random.seed(42)

# Streaming simulation: 3 phases
# Phase 1 (0-500): normal ~ N(0,1)
# Phase 2 (500-800): drift — mean shifts to 3
# Phase 3 (800-1200): new normal ~ N(3, 1.5)
n1, n2, n3 = 500, 300, 400
stream_data = np.concatenate([
    np.random.normal(0, 1, (n1, 4)),
    np.random.normal(0, 1, (n2, 4)) + np.linspace(0, 3, n2).reshape(-1, 1),  # gradual drift
    np.random.normal(3, 1.5, (n3, 4)),
])
# Inject point anomalies
anom_idx = np.random.choice(len(stream_data), 60, replace=False)
stream_data[anom_idx] += np.random.randn(60, 4) * 5
y_stream = np.zeros(len(stream_data), dtype=int)
y_stream[anom_idx] = 1
print(f"Stream: {len(stream_data)} points, {y_stream.sum()} anomalies, 3 phases")

In [ ]:
class StreamingAnomalyDetector:
    """
    Online Isolation Forest with sliding window retraining.
    Refits the model when the buffer fills up.
    """
    def __init__(self, window_size=200, refit_every=50, contamination=0.05):
        self.window_size   = window_size
        self.refit_every   = refit_every
        self.contamination = contamination
        self.buffer        = deque(maxlen=window_size)
        self.model         = None
        self.scores_       = []
        self.n_refits_     = 0
        self._since_refit  = 0

    def update(self, x):
        self.buffer.append(x)
        self._since_refit += 1
        if len(self.buffer) >= 50 and self._since_refit >= self.refit_every:
            self.model = IsolationForest(n_estimators=100, contamination=self.contamination, random_state=0)
            self.model.fit(np.array(self.buffer))
            self._since_refit = 0
            self.n_refits_ += 1
        if self.model is not None:
            score = -self.model.score_samples([x])[0]
        else:
            score = 0.0
        self.scores_.append(score)
        return score

detector = StreamingAnomalyDetector(window_size=200, refit_every=50)
for x in stream_data:
    detector.update(x)

scores = np.array(detector.scores_)
# Evaluate only after warmup
warmup = 200
valid = np.arange(warmup, len(stream_data))
auc = roc_auc_score(y_stream[valid], scores[valid])
print(f"Streaming IF ROC-AUC (post-warmup): {auc:.4f}")
print(f"Number of model refits: {detector.n_refits_}")

In [ ]:
# ADWIN drift detector
class ADWIN:
    """
    Adaptive Windowing drift detector.
    Triggers when the mean of two sub-windows diverges beyond a confidence bound.
    Reference: Bifet & Gavaldà (2007).
    """
    def __init__(self, delta=0.002):
        self.delta = delta
        self.window = deque()
        self.drift_detected_ = False
        self._drift_points = []

    def update(self, value, t=None):
        self.window.append(value)
        self.drift_detected_ = False
        w = list(self.window)
        n = len(w)
        if n < 4:
            return
        # Check all split points
        total_mean = np.mean(w)
        for split in range(1, n):
            w0 = np.array(w[:split]); w1 = np.array(w[split:])
            n0, n1 = len(w0), len(w1)
            m0, m1 = w0.mean(), w1.mean()
            # Hoeffding bound
            epsilon_cut = np.sqrt(
                (1 / (2 * n0) + 1 / (2 * n1)) * np.log(4 * n / self.delta)
            )
            if abs(m0 - m1) >= epsilon_cut:
                # Drift detected: shrink window to the newer sub-window
                for _ in range(split):
                    if self.window:
                        self.window.popleft()
                self.drift_detected_ = True
                if t is not None:
                    self._drift_points.append(t)
                break

# Run ADWIN on the first feature's values
adwin = ADWIN(delta=0.002)
drift_points = []
feature0 = stream_data[:, 0]
for t, val in enumerate(feature0):
    adwin.update(val, t)
    if adwin.drift_detected_:
        drift_points.append(t)

print(f"ADWIN detected {len(drift_points)} drift events at: {drift_points[:10]}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(feature0, color="steelblue", lw=0.7, alpha=0.8, label="Feature 0")
for dp in drift_points:
    axes[0].axvline(dp, color="red", alpha=0.4, lw=1.5)
axes[0].axvline(n1, color="black", linestyle="--", lw=1, label="True drift start")
axes[0].axvline(n1+n2, color="black", linestyle=":", lw=1, label="Drift end")
axes[0].set_title("Feature 0 stream with ADWIN drift events (red)")
axes[0].legend(fontsize=8)

axes[1].plot(scores, color="darkorange", lw=0.7, alpha=0.8, label="Anomaly score")
axes[1].scatter(np.where(y_stream==1)[0], scores[y_stream==1], c="red", s=20, zorder=5, label="True anomaly")
axes[1].axvline(n1, color="black", linestyle="--", lw=1)
axes[1].axvline(n1+n2, color="black", linestyle=":", lw=1)
axes[1].set_title("Streaming IF anomaly score over time")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/06_streaming.png", dpi=120)
plt.show()